# Part 1: Build knowledge bases with agentic retrieval

Part 1 at Caldova begins with a procurement analyst investigating supplier risk. You need one knowledge system that can retrieve purchasing policy, supplier evidence, and approved sourcing commitments without blurring the role of each source.

**🎯 Mission**
- Upload Caldova's purchasing policy directly to a **File Knowledge Source** and query its controls
- Connect the **supplier evidence** and **sourcing documents** indexes as **Indexed Knowledge Sources**
- Build a multi-source **Knowledge Base** that routes procurement questions to the right evidence

## Architecture at a glance

[![Part 1 target architecture, not live deployment status.](../../img/architecture/part1-before.png)](../../img/architecture/part1-before.png)

Orange highlights the policy-only build. Prepared indexes are faded, and later Fabric and Work IQ work stays covered.

## The building blocks

In Foundry IQ (Azure AI Search), a **knowledge base** is a top-level resource that connects a chat completion model to one or more knowledge sources. It defines which data sources to query, which model to use for reasoning, and how query execution should be optimized.

In this notebook, you'll explore two types of knowledge sources:

1. **File Knowledge Source**: Upload raw files directly. The service handles chunking, embedding, and indexing automatically.
2. **Indexed Knowledge Sources**: Connect to pre-built search indexes for production workloads with full control over schema and processing.

## Pre-configured environment

This lab uses the following Azure resources, already set up for you:

* Foundry IQ (Azure AI Search)
  * `supplier-evidence` index: 31 chunks from invoices, shipment evidence, production yield records, and quality records
  * `sourcing-documents` index: 16 chunks from RFP responses, agreements, amendments, and purchase orders
  * Semantic ranking enabled with pre-computed embeddings
* Azure OpenAI from Microsoft Foundry Models
  * `gpt-5.4`: Chat completion for reasoning and synthesis
  * `text-embedding-3-large`: Embedding model for vector search
* Microsoft Fabric

## Step 1: Set up environment variables

Load the configuration for your Azure resources. The `.env` file in the project folder has everything you need: Foundry IQ (Azure AI Search) endpoints, Azure OpenAI credentials, and model configuration.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

### Section map

![Part 1, step 1: Load existing service addresses and model names.](../../img/architecture/step-configuration.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Architecture for the start phase.](../../img/architecture/azure-start.svg)

**Architecture in brief:** Search, model deployments, indexed documents and remote data already exist in the lab. The notebook is a local client. The attendee has not created a knowledge base yet.

**Step by step:**

1. Locate the participant device and load the notebook environment variables. This configures a local client, not a new Azure deployment.
2. Follow the dashed gray arrow to Entra ID and sign in. Keep tokens and environment-specific identifiers out of shared files.
3. Inspect the preconfigured Azure AI Search service, its two indexes, and the Foundry model deployments.
4. Locate the separate Fabric workspace and Microsoft 365 boundary. Their data and identity prerequisites are prepared outside the notebook.
5. Notice the gray KB and source cards. No attendee KB exists at this point, so no blue retrieval or purple KB model-call arrows are shown.

This is the target, not live deployment status. Inspect actual configuration and evidence.

</details>


In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"

SUPPLIER_EVIDENCE_INDEX = "supplier-evidence"
SOURCING_DOCUMENTS_INDEX = "sourcing-documents"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")

### Compare with the product

**Expected after this section:** Load existing service addresses and model names.

**Where to look:** Local configuration.

**What to verify:** Never print credentials.

This is inspection guidance, not proof of a successful run.


### Concept: Two models, two jobs

Before configuring the file source, separate the embedding deployment from the chat deployment. The embedding model turns text into vectors for retrieval. It does not write an answer. The chat model helps plan retrieval and write answers. Both model deployments already exist.

[![Chat model produces language, while the embedding model represents related meaning as vectors.](../../img/concepts/model-roles.png)](../../img/concepts/model-roles.png)

*Concept illustration, not a portal screenshot or a successful query result.*


---

## A: File Knowledge Source

A **File Knowledge Source** lets you upload raw files directly. The service handles chunking, embedding, and indexing automatically. This is the fastest way to get documents into a knowledge base.

## Step 2: Create a File Knowledge Source

Configure a file knowledge source with an embedding model for automatic ingestion. The `ingestion_parameters` specify:
- **Embedding model**: Generates vector embeddings for the uploaded content
- **Content extraction mode**: How to extract text from documents

### Section map

![Part 1, step 2: Create a file source.](../../img/architecture/step-file-source.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Architecture for the policy phase.](../../img/architecture/azure-policy.svg)

**Architecture in brief:** Part 1 uploads the purchasing policy and connects it to a dedicated Search knowledge base. The KB retrieves policy evidence and calls the existing model to return a cited answer.

**Step by step:**

1. Configure the local notebook and sign in through Entra ID. The dashed gray identity arrow represents authentication.
2. Create the file knowledge source and upload the purchasing policy. Follow the dashed gray connections from the PDF to the file source and from the source to the embedding model.
3. Wait for ingestion and inspect the uploaded file status. A successful upload alone does not prove that the policy is searchable.
4. Create the policy KB with the file source and the existing synthesis model. Follow the solid blue notebook-to-KB arrow when you submit a question.
5. Follow the blue KB-to-file-source arrow for evidence and the purple KB-to-model arrow for planning and answer synthesis.
6. Read the cited policy answer in the notebook and verify its file reference and source activity. The response returns on the same connections.

This is the target, not live deployment status. Inspect actual configuration and evidence.

</details>


In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    FileKnowledgeSource,
    FileKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "file-docs-knowledge-source"

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

file_ks = FileKnowledgeSource(
    name=FILE_KNOWLEDGE_SOURCE_NAME,
    description="Caldova purchasing policy covering approval thresholds, sourcing routes, supplier governance, and escalation responsibilities",
    file_parameters=FileKnowledgeSourceParameters(
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AZURE_OPENAI_ENDPOINT,
                    deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                    model_name=AZURE_OPENAI_EMBEDDING_MODEL,
                    api_key=AZURE_OPENAI_KEY,
                )
            ),
            content_extraction_mode="minimal",
            )
    ),
)

index_client.create_or_update_knowledge_source(file_ks)
print(f"File knowledge source '{FILE_KNOWLEDGE_SOURCE_NAME}' created or updated successfully.")

### Compare with the product

**Expected after this section:** Create a file source.

**Where to look:** Search > Agentic retrieval > Knowledge sources.

**What to verify:** Inspect source kind and embedding deployment.

This is inspection guidance, not proof of a successful run.


### Find this step in Azure portal

**Open:** Azure portal > your lab Search service > Agentic retrieval > Knowledge sources > file-docs-knowledge-source.

**Look for:** The subtitle identifies a **File** source. The **Index** link points to the index that the service manages for your uploaded policy. Under **Text vectorization model**, compare the deployment with the embedding settings in this notebook.

[![Original Azure portal view: Policy source, embedding configuration and uploaded files.](../../img/portal/policy-source-files.png)](../../img/portal/policy-source-files.png)

**How to read this:** This reference shows the source after upload too. Creating the source in this step does not itself upload the PDF.

*Reference capture: 2026-10-20. Gray blocks redact environment identifiers. Orange outlines are added annotations, not status indicators. This image is not a live check of your resources.*


### Concept: What happens after the upload?

The service prepares the policy as searchable passages with vectors and source metadata. This is ingestion, not an answer to a question. Wait for file processing and inspect errors before querying. The prepared supplier and sourcing indexes use a different path: you connect them later without uploading their PDFs again.

[![Policy upload leads to extraction, passage splitting, embeddings and an index. Existing indexes are connected separately.](../../img/concepts/document-ingestion.png)](../../img/concepts/document-ingestion.png)

*Concept illustration, not a portal screenshot or a successful query result.*


## Step 3: Upload files to the knowledge source

Upload a document directly to the file knowledge source. The service will automatically extract text, chunk it into searchable segments, and generate vector embeddings.

⏱️ The sample file takes around 40 seconds to ingest.

### Section map

![Part 1, step 3: Upload the policy.](../../img/architecture/step-ingestion.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Architecture for the policy phase.](../../img/architecture/azure-policy.svg)

**Architecture in brief:** Part 1 uploads the purchasing policy and connects it to a dedicated Search knowledge base. The KB retrieves policy evidence and calls the existing model to return a cited answer.

**Step by step:**

1. Configure the local notebook and sign in through Entra ID. The dashed gray identity arrow represents authentication.
2. Create the file knowledge source and upload the purchasing policy. Follow the dashed gray connections from the PDF to the file source and from the source to the embedding model.
3. Wait for ingestion and inspect the uploaded file status. A successful upload alone does not prove that the policy is searchable.
4. Create the policy KB with the file source and the existing synthesis model. Follow the solid blue notebook-to-KB arrow when you submit a question.
5. Follow the blue KB-to-file-source arrow for evidence and the purple KB-to-model arrow for planning and answer synthesis.
6. Read the cited policy answer in the notebook and verify its file reference and source activity. The response returns on the same connections.

This is the target, not live deployment status. Inspect actual configuration and evidence.

</details>


In [ ]:
from pathlib import Path

data_dir = Path("../..") / "data"
files_to_upload = [
    data_dir / "caldova" / "pdfs" / "CAL-POL-PUR-001.pdf"
]

uploaded_files = []
for file_path in files_to_upload:
    with open(file_path, "rb") as f:
        file_content = f.read()
    uploaded = index_client.upload_knowledge_source_file(
        FILE_KNOWLEDGE_SOURCE_NAME, file_content, filename=file_path.name
    )
    uploaded_files.append(uploaded)
    print(f"Uploaded: {file_path.name} (file_id: {uploaded.file_id}, {uploaded.file_size_bytes} bytes)")

print(f"\nTotal files uploaded: {len(uploaded_files)}")

### Compare with the product

**Expected after this section:** Upload the policy.

**Where to look:** Search > file source > Files.

**What to verify:** Check processing status and errors.

This is inspection guidance, not proof of a successful run.


### Find this step in Azure portal

**Open:** Azure portal > your lab Search service > Agentic retrieval > Knowledge sources > file-docs-knowledge-source > Files.

**Look for:** Find `CAL-POL-PUR-001.pdf` and inspect its **Status**. The original lab capture shows **Ready**.

[![Original Azure portal view: Policy source, embedding configuration and uploaded files.](../../img/portal/policy-source-files.png)](../../img/portal/policy-source-files.png)

**How to read this:** There are four rows because this lab was run repeatedly. Upload the policy once for your run, not four times. A Ready status is an ingestion check, not proof that the next query will retrieve it.

*Reference capture: 2026-10-20. Gray blocks redact environment identifiers. Orange outlines are added annotations, not status indicators. This image is not a live check of your resources.*


### Concept: Connect evidence, then coordinate retrieval

The source you created is a connection definition. This step creates the KB that references it. A KB can reuse source definitions without making another copy of the data. Useful source descriptions help the planner decide where to look. Later parts create separate KB variants.

[![An evidence location connects through a knowledge source to a knowledge base with its own retrieval settings.](../../img/concepts/source-versus-kb.png)](../../img/concepts/source-versus-kb.png)

*Concept illustration, not a portal screenshot or a successful query result.*


## Step 4: Create a knowledge base for the file source

A knowledge base is the orchestration layer that combines:

1. Your data sources (the file knowledge source)
2. An LLM (Azure OpenAI) for understanding queries and generating answers
3. Configuration for how to process queries and format responses

This knowledge base uses `ANSWER_SYNTHESIS` output mode, so the attached LLM will generate a complete natural language answer from the retrieved chunks.

### Section map

![Part 1, step 4: Create the policy-only KB.](../../img/architecture/step-file-source.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Architecture for the policy phase.](../../img/architecture/azure-policy.svg)

**Architecture in brief:** Part 1 uploads the purchasing policy and connects it to a dedicated Search knowledge base. The KB retrieves policy evidence and calls the existing model to return a cited answer.

**Step by step:**

1. Configure the local notebook and sign in through Entra ID. The dashed gray identity arrow represents authentication.
2. Create the file knowledge source and upload the purchasing policy. Follow the dashed gray connections from the PDF to the file source and from the source to the embedding model.
3. Wait for ingestion and inspect the uploaded file status. A successful upload alone does not prove that the policy is searchable.
4. Create the policy KB with the file source and the existing synthesis model. Follow the solid blue notebook-to-KB arrow when you submit a question.
5. Follow the blue KB-to-file-source arrow for evidence and the purple KB-to-model arrow for planning and answer synthesis.
6. Read the cited policy answer in the notebook and verify its file reference and source activity. The response returns on the same connections.

This is the target, not live deployment status. Inspect actual configuration and evidence.

</details>


In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalOutputMode,
)

FILE_KNOWLEDGE_BASE_NAME = "caldova-policy-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=FILE_KNOWLEDGE_BASE_NAME,
    description="Caldova purchasing policy knowledge base",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name=FILE_KNOWLEDGE_SOURCE_NAME),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{FILE_KNOWLEDGE_BASE_NAME}' created or updated successfully.")

### Compare with the product

**Expected after this section:** Create the policy-only KB.

**Where to look:** Search > Agentic retrieval > Knowledge bases.

**What to verify:** Confirm one source and the attached model.

This is inspection guidance, not proof of a successful run.


### Find this step in Azure portal

**Open:** Azure portal > your lab Search service > Agentic retrieval > Knowledge bases > caldova-policy-knowledge-base.

**Look for:** Under **Basics > Knowledge sources**, there is exactly one source: `file-docs-knowledge-source`.

[![Original Azure portal view: Policy-only knowledge base.](../../img/portal/policy-knowledge-base.png)](../../img/portal/policy-knowledge-base.png)

**How to read this:** A knowledge source is the connection to evidence. The knowledge base groups the connections used by a query. Compare the source name before continuing.

*Reference capture: 2026-10-20. Gray blocks redact environment identifiers. Orange outlines are added annotations, not status indicators. This image is not a live check of your resources.*


## Step 5: Query the file knowledge base

Ask a question about the uploaded document. The knowledge base will search across the file content and synthesize a natural language answer with citations.

### Section map

![Part 1, step 5: Retrieve policy evidence.](../../img/architecture/step-policy-query.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Architecture for the policy phase.](../../img/architecture/azure-policy.svg)

**Architecture in brief:** Part 1 uploads the purchasing policy and connects it to a dedicated Search knowledge base. The KB retrieves policy evidence and calls the existing model to return a cited answer.

**Step by step:**

1. Configure the local notebook and sign in through Entra ID. The dashed gray identity arrow represents authentication.
2. Create the file knowledge source and upload the purchasing policy. Follow the dashed gray connections from the PDF to the file source and from the source to the embedding model.
3. Wait for ingestion and inspect the uploaded file status. A successful upload alone does not prove that the policy is searchable.
4. Create the policy KB with the file source and the existing synthesis model. Follow the solid blue notebook-to-KB arrow when you submit a question.
5. Follow the blue KB-to-file-source arrow for evidence and the purple KB-to-model arrow for planning and answer synthesis.
6. Read the cited policy answer in the notebook and verify its file reference and source activity. The response returns on the same connections.

This is the target, not live deployment status. Inspect actual configuration and evidence.

</details>


In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)
from IPython.display import Markdown, display

file_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=FILE_KNOWLEDGE_BASE_NAME, credential=search_credential
)

file_ks_params = FileKnowledgeSourceParams(
    knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = "What approval thresholds, sourcing routes, and escalation responsibilities does Caldova's purchasing policy define?"

req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(text=question)
            ],
        )
    ],
    knowledge_source_params=[file_ks_params],
    include_activity=True,
)

result = file_kb_client.retrieve(retrieval_request=req)
if not result.response or not result.response[0].content:
    raise RuntimeError("Knowledge base returned no answer content.")
answer_text = getattr(result.response[0].content[0], "text", None)
if not isinstance(answer_text, str):
    raise TypeError("Knowledge base returned an unsupported answer format.")
display(Markdown(answer_text))

### Review the activity log

The activity log shows the sequence of steps taken by the knowledge base. Each activity contains:

* `type`: The type of activity. For this log, you will see the first step of "modelQueryPlanning", multiple "file" steps (one for each query sent to the file knowledge source), "modelAnswerSynthesis" for the answer synthesis step, and "agenticReasoning" for the semantic re-ranking step.
* `elapsedMS`: The duration of the activity.
* `id`: The activity ID, which is helpful to connect each reference to the activity that produced it. You'll review references next.

The activity entries also contain other metadata specific to their type, so that you can see how many tokens were used by any model-powered steps and see the queries sent to sources for retrieval steps.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Concept: How to read the references

When an answer uses a marker such as `[ref:3]`, find the matching reference `id`. Then use `activitySource` to locate the corresponding activity `id`. For indexed references, inspect document and page metadata when supplied. The first query here uses a file source, so its reference fields can differ from the later indexed examples.

[![An answer marker points to a reference, which links to source activity and the underlying evidence.](../../img/concepts/citation-trail.png)](../../img/concepts/citation-trail.png)

*Concept illustration, not a portal screenshot or a successful query result.*


### Review the references

The next cell prints the raw `references` array returned by the knowledge base.

The references show which sources were retrieved, and the source metadata for each reference. Every result contains:

* `type`: Always "file", since this knowledge base only contains a file source.
* `id`: A numeric identifier for the reference. This is what appears in the answer as citation markers like `[ref:1]`.
* `activitySource`: The id of the retrieval activity in the activity log that produced this reference. You can use this to trace which search step returned this document.
* `sourceData`: Fields from the retrieved chunk in the auto-created search index - always `uid`, `snippet`, and `metadata_storage_path` for file source indexes.
* `rerankerScore`: A score between 0-4 from the re-ranking model, where 4 is the best.
* `docName`: The auto-generated filename for the uploaded file.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

### Compare with the product

**Expected after this section:** Retrieve policy evidence.

**Where to look:** Notebook answer, activity and references.

**What to verify:** Verify the cited policy passage supports the answer.

This is inspection guidance, not proof of a successful run.


### Concept: What is inside a prepared index?

A chunk is one searchable passage, not necessarily one PDF. A record can contain text, a vector and citation metadata. Keyword retrieval helps with exact IDs, while vector retrieval helps with related wording. Inspect the actual schema and counts below rather than treating the illustration as a sample result.

[![An illustrative passage contains text, vector, citation metadata and identifiers, with keyword and vector retrieval paths.](../../img/concepts/index-anatomy.png)](../../img/concepts/index-anatomy.png)

*Concept illustration, not a portal screenshot or a successful query result.*


---

## B: Indexed knowledge sources

**Indexed Knowledge Sources** connect to pre-built search indexes, giving you full control over schema, chunking, and embedding. This is ideal for production workloads with large datasets.

For this section, two restored search indexes already contain document chunks. `supplier-evidence` holds operational evidence such as invoices, shipment records, yield records, and quality findings. `sourcing-documents` holds approved procurement records such as RFP responses, agreements, amendments, and purchase orders.

## Step 6: Verify search indexes

Run the cell below to confirm each restored index has the expected number of chunks:

* `supplier-evidence`: 31 document chunks
* `sourcing-documents`: 16 document chunks

### Section map

![Part 1, step 6: Inspect the prepared indexes.](../../img/architecture/step-indexes.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Architecture for the documents phase.](../../img/architecture/azure-documents.svg)

**Architecture in brief:** A separate Search knowledge base connects policy with two preloaded document indexes. The KB combines the three evidence sources and uses its model to return a cited answer.

**Step by step:**

1. Reuse the ingested policy and the preloaded supplier-evidence and sourcing-documents indexes. Do not upload those indexed documents again.
2. Create the two search-index knowledge source definitions, then create a separate KB connecting all three sources.
3. Submit a question from the local notebook. Follow the solid blue connection to this three-source KB.
4. Follow the blue connections from the KB to the policy source and the existing indexes. The indexes box represents two distinct document sources.
5. Follow the purple connection to Azure OpenAI for planning and a grounded, cited answer. The dashed gray embedding connection belongs to policy ingestion, not every query.
6. Check activity and references to see which sources actually supplied evidence. Configuration alone does not prove that all three were used.

This is the target, not live deployment status. Inspect actual configuration and evidence.

</details>


In [ ]:
from azure.search.documents import SearchClient

for index_name in [SUPPLIER_EVIDENCE_INDEX, SOURCING_DOCUMENTS_INDEX]:
    client = SearchClient(
        endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
        index_name=index_name,
        credential=search_credential,
    )
    count = client.get_document_count()
    print(f"{index_name}: {count} document chunks")
    client.close()

### Compare with the product

**Expected after this section:** Inspect the prepared indexes.

**Where to look:** Search > Search management > Indexes.

**What to verify:** Check actual schemas and nonzero counts.

This is inspection guidance, not proof of a successful run.


### Find this step in Azure portal

**Open:** Azure portal > your lab Search service > Search management > Indexes.

**Look for:** Find the prepared `supplier-evidence` and `sourcing-documents` indexes and their nonzero **Document count**. The file source also has a generated index.

[![Original Azure portal view: Existing search indexes and document counts.](../../img/portal/search-indexes.png)](../../img/portal/search-indexes.png)

**How to read this:** The captured counts are 31 and 16 for the prepared indexes. Counts can change between runs. This step inspects existing data rather than creating the two indexes.

*Reference capture: 2026-10-20. Gray blocks redact environment identifiers. Orange outlines are added annotations, not status indicators. This image is not a live check of your resources.*


## Step 7: Create two indexed knowledge sources

Create one knowledge source for each restored index:

* `supplier-evidence-knowledge-source`: Supplier invoices, shipment evidence, production yield records, and quality records
* `sourcing-documents-knowledge-source`: RFP responses, agreements, amendments, and purchase orders

Both sources search the `chunk` field and return `blob_path`, `title`, `page_number_from`, and `page_number_to` as citation source data. These fields match the portable Caldova index schema.

### Section map

![Part 1, step 7: Connect the two prepared indexes.](../../img/architecture/step-indexes.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Architecture for the documents phase.](../../img/architecture/azure-documents.svg)

**Architecture in brief:** A separate Search knowledge base connects policy with two preloaded document indexes. The KB combines the three evidence sources and uses its model to return a cited answer.

**Step by step:**

1. Reuse the ingested policy and the preloaded supplier-evidence and sourcing-documents indexes. Do not upload those indexed documents again.
2. Create the two search-index knowledge source definitions, then create a separate KB connecting all three sources.
3. Submit a question from the local notebook. Follow the solid blue connection to this three-source KB.
4. Follow the blue connections from the KB to the policy source and the existing indexes. The indexes box represents two distinct document sources.
5. Follow the purple connection to Azure OpenAI for planning and a grounded, cited answer. The dashed gray embedding connection belongs to policy ingestion, not every query.
6. Check activity and references to see which sources actually supplied evidence. Configuration alone does not prove that all three were used.

This is the target, not live deployment status. Inspect actual configuration and evidence.

</details>


In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME = "supplier-evidence-knowledge-source"
SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME = "sourcing-documents-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    FILE_KNOWLEDGE_SOURCE_NAME,
    SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
    SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (
        SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
        SUPPLIER_EVIDENCE_INDEX,
        "Caldova supplier invoices, shipment evidence, production yield records, and quality records",
    ),
    (
        SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
        SOURCING_DOCUMENTS_INDEX,
        "Caldova RFP responses, agreements, amendments, and purchase orders",
    ),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="title"),
                SearchIndexFieldReference(name="page_number_from"),
                SearchIndexFieldReference(name="page_number_to"),
            ],
            search_fields=[SearchIndexFieldReference(name="chunk")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
    print(f"Knowledge source '{knowledge_source.name}' created or updated successfully.")

### Compare with the product

**Expected after this section:** Connect the two prepared indexes.

**Where to look:** Search > Agentic retrieval > Knowledge sources.

**What to verify:** Compare each source with its target index.

This is inspection guidance, not proof of a successful run.


### Find this step in Azure portal

**Open:** Azure portal > your lab Search service > Agentic retrieval > Knowledge sources > select each indexed source.

**Look for:** Compare the source name at the top with the **Index** field. `supplier-evidence-knowledge-source` points to `supplier-evidence`, and `sourcing-documents-knowledge-source` points to `sourcing-documents`.

[![Original Azure portal view: Supplier evidence indexed knowledge source.](../../img/portal/supplier-index-source.png)](../../img/portal/supplier-index-source.png)

[![Original Azure portal view: Sourcing documents indexed knowledge source.](../../img/portal/sourcing-index-source.png)](../../img/portal/sourcing-index-source.png)

**How to read this:** These definitions tell a KB which existing index to use. They do not duplicate the indexed documents. Inspect the settings without changing or saving them.

*Reference capture: 2026-10-20. Gray blocks redact environment identifiers. Orange outlines are added annotations, not status indicators. This image is not a live check of your resources.*


## Step 8: Create the multi-source knowledge base

Create a knowledge base that contains the purchasing policy file source and both search index knowledge sources. It uses `output_mode=ANSWER_SYNTHESIS`, so the attached model can answer the original query with grounded citations while keeping each evidence type traceable to its source.

### Section map

![Part 1, step 8: Create a separate three-source KB.](../../img/architecture/step-indexes.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Architecture for the documents phase.](../../img/architecture/azure-documents.svg)

**Architecture in brief:** A separate Search knowledge base connects policy with two preloaded document indexes. The KB combines the three evidence sources and uses its model to return a cited answer.

**Step by step:**

1. Reuse the ingested policy and the preloaded supplier-evidence and sourcing-documents indexes. Do not upload those indexed documents again.
2. Create the two search-index knowledge source definitions, then create a separate KB connecting all three sources.
3. Submit a question from the local notebook. Follow the solid blue connection to this three-source KB.
4. Follow the blue connections from the KB to the policy source and the existing indexes. The indexes box represents two distinct document sources.
5. Follow the purple connection to Azure OpenAI for planning and a grounded, cited answer. The dashed gray embedding connection belongs to policy ingestion, not every query.
6. Check activity and references to see which sources actually supplied evidence. Configuration alone does not prove that all three were used.

This is the target, not live deployment status. Inspect actual configuration and evidence.

</details>


In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "caldova-supply-chain-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Caldova knowledge base for purchasing policy, supplier evidence, and approved sourcing commitments",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated successfully.")

### Compare with the product

**Expected after this section:** Create a separate three-source KB.

**Where to look:** Search > Agentic retrieval > Knowledge bases.

**What to verify:** Confirm policy and both indexed sources.

This is inspection guidance, not proof of a successful run.


### Find this step in Azure portal

**Open:** Azure portal > your lab Search service > Agentic retrieval > Knowledge bases > caldova-supply-chain-knowledge-base.

**Look for:** Count three source cards: purchasing policy, supplier evidence, and sourcing documents.

[![Original Azure portal view: Three-source document knowledge base.](../../img/portal/three-source-knowledge-base.png)](../../img/portal/three-source-knowledge-base.png)

**How to read this:** This is a separate KB from `caldova-policy-knowledge-base`. The earlier policy-only KB still exists. Active source cards do not prove that a particular query used every source.

*Reference capture: 2026-10-20. Gray blocks redact environment identifiers. Orange outlines are added annotations, not status indicators. This image is not a live check of your resources.*


## Step 9: Query the multi-source knowledge base

Ask an operational question that should retrieve excursion and disposition evidence from `supplier-evidence`:

* _"Shipment SHIP-VC-1026-BOS-FRA exceeded its temperature range. What happened, and can the 4,800 units be distributed?"_

The knowledge base also retains the purchasing policy file and approved sourcing documents. After the sample query, try these questions:

* _"Compare Summit Dose's documented CALD-201 commitments across its RFP response, agreement amendment, and purchase order."_
* _"Was lot LOT-MD-1026-A released, and should Caldova pay the $125,600 yield adjustment?"_

Use the activity log and references to verify which source answered each question. Keep policy controls, operational supplier evidence, and approved sourcing commitments distinct in your conclusions.

### Section map

![Part 1, step 9: Retrieve across document roles.](../../img/architecture/step-documents-query.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Architecture for the documents phase.](../../img/architecture/azure-documents.svg)

**Architecture in brief:** A separate Search knowledge base connects policy with two preloaded document indexes. The KB combines the three evidence sources and uses its model to return a cited answer.

**Step by step:**

1. Reuse the ingested policy and the preloaded supplier-evidence and sourcing-documents indexes. Do not upload those indexed documents again.
2. Create the two search-index knowledge source definitions, then create a separate KB connecting all three sources.
3. Submit a question from the local notebook. Follow the solid blue connection to this three-source KB.
4. Follow the blue connections from the KB to the policy source and the existing indexes. The indexes box represents two distinct document sources.
5. Follow the purple connection to Azure OpenAI for planning and a grounded, cited answer. The dashed gray embedding connection belongs to policy ingestion, not every query.
6. Check activity and references to see which sources actually supplied evidence. Configuration alone does not prove that all three were used.

This is the target, not live deployment status. Inspect actual configuration and evidence.

</details>


In [ ]:
from azure.search.documents.knowledgebases.models import (
    SearchIndexKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential,
)

supplier_evidence_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)
sourcing_documents_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = (
    "Shipment SHIP-VC-1026-BOS-FRA exceeded its temperature range. "
    "What happened, and can the 4,800 units be distributed?"
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[file_ks_params, supplier_evidence_params, sourcing_documents_params],
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
if not result.response or not result.response[0].content:
    raise RuntimeError("Knowledge base returned no answer content.")
answer_text = getattr(result.response[0].content[0], "text", None)
if not isinstance(answer_text, str):
    raise TypeError("Knowledge base returned an unsupported answer format.")
display(Markdown(answer_text))

### Review the activity log

For this activity log, you may see a `file` step for purchasing policy and `searchIndex` steps for queries sent to the supplier evidence or sourcing documents indexes.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

The references identify which knowledge source supported each claim. Indexed references have `type: "searchIndex"` and include citation metadata such as `title`, `blob_path`, and page numbers. If purchasing policy is relevant, you may also see a reference with `type: "file"`.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

Look through the references and activity log above. Can you find:

1. Which supplier evidence explains the temperature excursion and its root cause?
2. Which evidence records the quarantine actions and final distribution decision for the 4,800 units?
3. Do indexed references expose `title`, `blob_path`, `page_number_from`, and `page_number_to`?
4. Did the knowledge base keep operational shipment evidence distinct from purchasing policy and approved sourcing commitments?

## Bonus: Query from the GitHub Copilot App

Every knowledge base exposes an MCP server endpoint, which can be added to any MCP-compatible client like VS Code or the GitHub Copilot App. Run the cell below to print the MCP URL for the Caldova knowledge base, then follow the steps after it to query the knowledge base from the GitHub Copilot App.

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
print(mcp_url)

### Sign in to GitHub

Sign in through the [Skillable Events GitHub enterprise](https://github.com/enterprises/skillable-events/sso).

You will be signed into a special account created for the lab environment, not your actual GitHub account.

### Start the GitHub Copilot App

1. Open the GitHub Copilot App by finding the shortcut on the desktop.
2. Select "Sign in to GitHub" button.
3. When the browser pops up, paste the device code that was automatically copied into your clipboard, and select "Authorize github".
4. Return to the app. You don't need to connect any repositories, so select "Continue" on repository selection screen and "Finish" on the preferences screen.


### Add the knowledge base MCP server

1. Select **Customize** in the sidebar.
2. Select the **MCP** tab.
3. Open the **Add** menu, then select **MCP server**.
4. Fill in the details for the Caldova knowledge base:
   * Server name: `caldova-kb`
   * Server type: **HTTP** (*not* Local/SSE)
   * URL: Paste the URL printed by the cell above.
   * Headers: Enter `api-key` for the header name and the value of `AZURE_SEARCH_ADMIN_KEY` from your `.env` file for its value.
5. Leave other fields at their defaults, then select **Add server**.
6. Confirm that `caldova-kb` appears in the list of MCP servers.

### Ask a question grounded by the knowledge base

Create a new chat by selecting **+** next to **Chats** in the sidebar, then ask a question that matches this notebook. For example:

_"Compare Summit Dose's documented CALD-201 commitments across its Caldova RFP response, agreement amendment, and purchase order."_

### Compare with the product

**Expected after this section:** Retrieve across document roles.

**Where to look:** Notebook answer, activity and references.

**What to verify:** Separate operational facts from policy and commitments.

This is inspection guidance, not proof of a successful run.


## Architecture recap: what Part 1 created

[![Part 1 architecture: the knowledge base connects the purchasing policy with two prepared document indexes.](../../img/architecture/part1-after.png)](../../img/architecture/part1-after.png)

This is the target wiring for the notebook. The indexes and model are prepared for you. The knowledge sources and knowledge base connect them so answers can be grounded in policy and procurement evidence.

## ✅ Mission complete

**What you built:**

* ✓ **File Knowledge Source**: Caldova's purchasing policy uploaded directly for approval, sourcing-route, governance, and escalation questions.
* ✓ **Indexed Knowledge Sources**: Portable supplier evidence and sourcing document indexes using the `chunk` field and citation metadata.
* ✓ **Multi-source Knowledge Base**: One Caldova KB that keeps policy, operational evidence, and approved sourcing commitments traceable to their sources.

➡️ Continue to [Part 2: Add structured data from Fabric IQ](part2-fabric-iq-to-kb.ipynb).